In [1]:
import pandas as pd

data = pd.read_csv("../Data/rtg_C.csv")
print(data.head(10))
print(data.columns)
print(data.shape)

   att0  att1  att2  att3  att4  att5  att6  class
0     1     0     0     0     0     0     0      0
1     0     1     1     1     1     0     0      0
2     1     0     1     2     1     1     1      0
3     1     0     0     3     1     0     0      1
4     0     0     1     4     1     0     1      1
5     1     1     1     5     0     0     1      1
6     0     1     0     6     1     1     0      1
7     1     0     0     7     1     1     0      0
8     1     1     0     8     0     0     0      0
9     1     1     1     9     1     0     0      0
Index(['att0', 'att1', 'att2', 'att3', 'att4', 'att5', 'att6', 'class'], dtype='object')
(50, 8)


In [2]:
from collections import Counter
import math

def entropy(labels):

    counts = Counter(labels)

    total = len(labels)

    ent = 0

    for c in counts.values():

        p = c / total

        ent -= p * math.log2(p)

    return ent

In [3]:
def information_gain(data, labels, feature):

    parent_entropy = entropy(labels)

    values = set(data[:, feature])

    weighted_entropy = 0

    total = len(labels)

    for v in values:

        mask = data[:, feature] == v

        subset_labels = labels[mask]

        weighted_entropy += (len(subset_labels)/total) * entropy(subset_labels)

    return parent_entropy - weighted_entropy

In [4]:
def best_split(data, labels):

    best_feature = None
    best_ig = -1

    num_features = data.shape[1]

    for feature in range(num_features):

        ig = information_gain(data, labels, feature)

        if ig > best_ig:
            best_ig = ig
            best_feature = feature

    return best_feature, best_ig

In [5]:
class Node:

    def __init__(self):
        self.feature = None
        self.children = {}
        self.is_leaf = False
        self.class_counts = None
        self.entropy = None
        self.ig = None

In [6]:
from collections import Counter
import numpy as np

def build_tree(data, labels, remaining_features):

    node = Node()

    # Store node information
    node.entropy = entropy(labels)
    node.class_counts = dict(Counter(labels))

    # Stop if all samples belong to one class
    if node.entropy == 0:
        node.is_leaf = True
        return node

    # Stop if there are no features left
    if len(remaining_features) == 0:
        node.is_leaf = True
        return node

    # Find the best feature among the remaining features
    best_feature = None
    best_ig = -1

    for feature in remaining_features:

        ig = information_gain(data, labels, feature)

        if ig > best_ig:
            best_ig = ig
            best_feature = feature

    # Stop if Information Gain is too small
    if best_ig < 0.00001:
        node.is_leaf = True
        return node

    node.feature = best_feature
    node.ig = best_ig

    # Remaining features for the children
    new_remaining = remaining_features.copy()
    new_remaining.remove(best_feature)

    # Split the data
    values = np.unique(data[:, best_feature])

    for value in values:

        mask = data[:, best_feature] == value

        subset_data = data[mask]
        subset_labels = labels[mask]

        child = build_tree(subset_data, subset_labels, new_remaining)

        node.children[value] = child

    return node

In [7]:
# Features and labels
X = data.iloc[:, :-1].to_numpy()
y = data.iloc[:, -1].to_numpy()

for feature in range(X.shape[1]):

    ig = information_gain(X, y, feature)

    split_entropy = entropy(y) - ig

    print(
        f"Feature {feature}: Entropy={split_entropy:.6f}, IG={ig:.6f}"
    )
# List of feature indices
remaining_features = list(range(X.shape[1]))

# Build the decision tree
tree = build_tree(X, y, remaining_features)

print("Decision tree built successfully!")

Feature 0: Entropy=0.876005, IG=0.028376
Feature 1: Entropy=0.865463, IG=0.038918
Feature 2: Entropy=0.902198, IG=0.002183
Feature 3: Entropy=0.000000, IG=0.904381
Feature 4: Entropy=0.845908, IG=0.058474
Feature 5: Entropy=0.865463, IG=0.038918
Feature 6: Entropy=0.819087, IG=0.085294
Decision tree built successfully!


In [8]:
def print_tree(node, depth=0, edge_value=None):

    indent = "    " * depth

    if node.is_leaf:
        print(f"{indent}leaf {node.class_counts}")
        return
    
    print(f"{indent}feature {node.feature} (IG: {node.ig:.6f}, Entropy: {node.entropy:.6f})")

    for value, child in node.children.items():
        print(f"{indent}-- feature {node.feature} = {value} --")
        print_tree(child, depth + 1, value)

print_tree(tree)

feature 3 (IG: 0.904381, Entropy: 0.904381)
-- feature 3 = 0 --
    leaf {np.int64(0): 1}
-- feature 3 = 1 --
    leaf {np.int64(0): 1}
-- feature 3 = 2 --
    leaf {np.int64(0): 1}
-- feature 3 = 3 --
    leaf {np.int64(1): 1}
-- feature 3 = 4 --
    leaf {np.int64(1): 1}
-- feature 3 = 5 --
    leaf {np.int64(1): 1}
-- feature 3 = 6 --
    leaf {np.int64(1): 1}
-- feature 3 = 7 --
    leaf {np.int64(0): 1}
-- feature 3 = 8 --
    leaf {np.int64(0): 1}
-- feature 3 = 9 --
    leaf {np.int64(0): 1}
-- feature 3 = 10 --
    leaf {np.int64(1): 1}
-- feature 3 = 11 --
    leaf {np.int64(0): 1}
-- feature 3 = 12 --
    leaf {np.int64(1): 1}
-- feature 3 = 13 --
    leaf {np.int64(0): 1}
-- feature 3 = 14 --
    leaf {np.int64(1): 1}
-- feature 3 = 15 --
    leaf {np.int64(0): 1}
-- feature 3 = 16 --
    leaf {np.int64(0): 1}
-- feature 3 = 17 --
    leaf {np.int64(0): 1}
-- feature 3 = 18 --
    leaf {np.int64(0): 1}
-- feature 3 = 19 --
    leaf {np.int64(1): 1}
-- feature 3 = 20 --
    l

In [9]:
def save_tree(node, file, depth=0, edge_value=None):

    indent = "    " * depth

    # Leaf node
    if node.is_leaf:
        counts = {int(k): int(v) for k, v in node.class_counts.items()}
        file.write(f"{indent}leaf {counts}\n")
        return

    # Split node
    file.write(f"{indent}feature {node.feature}(")
    file.write(f"IG: {node.ig:.6f},")
    file.write(f"Entropy: {node.entropy:.6f})\n")

    # Children
    for value, child in node.children.items():
        file.write(f"{indent}-- feature {node.feature} = {value} --\n")
        save_tree(child, file, depth + 1, value)
with open("output_tree_C.txt", "w") as f:
    save_tree(tree, f)

In [10]:
def predict(node, sample):

    # If leaf node, return the majority class
    if node.is_leaf:
        return max(node.class_counts, key=node.class_counts.get)

    # Get the value of the splitting feature
    value = sample[node.feature]

    # Follow the correct branch
    child = node.children[value]

    return predict(child, sample)

In [11]:
def accuracy(tree, X, y):

    correct = 0

    for i in range(len(X)):

        prediction = predict(tree, X[i])

        if prediction == y[i]:
            correct += 1

    return correct / len(y)

In [12]:
acc = accuracy(tree, X, y)

print(f"Training Accuracy: {acc * 100:.2f}%")

Training Accuracy: 100.00%
